# Project 6: Agentic AI System - Warehouse Operations Support Agent (WOSA)

## Task 1: System Scope and Goal
The goal of **WOSA** is to assist warehouse engineers in diagnosing and resolving discrepancies between digital twin simulations and physical warehouse environments (Sim-to-Real gaps). 

### System Boundaries:
- **Task:** Autonomous diagnostic of robotic failures and inventory mismatches.
- **Decision Logic:** Uses a ReAct (Reasoning + Acting) loop to determine necessary diagnostic steps.
- **Tools:** Inventory Lookup, Robot Telemetry, and Physics Calibration.
- **Safeguards:** High-risk actions (e.g., system restarts) require manual human approval.

## Task 2: Agent Architecture
WOSA is designed as a single-agent system with the following components:
1. **Persona:** Logistics Operations Safety Coordinator.
2. **Reasoning Loop:** ReAct pattern (Thought -> Action -> Observation).
3. **Memory:** Conversation Buffer to maintain the state of current investigations.
4. **Tool Use:** Interface for Digital Twin and Physical Sensor APIs.

In [ ]:
import time
import json

# --- Task 3: Tool Implementation ---

def get_inventory_status(item_id: str) -> str:
    """Retrieve item count from the warehouse database."""
    # Mock Database
    inventory_db = {"PKG-001": 15, "PKG-002": 0, "PKG-003": 120}
    status = inventory_db.get(item_id, "Unknown")
    return f"RESULT: Inventory for {item_id} is {status} units."

def get_robot_diagnostics(robot_id: str) -> str:
    """Check battery, location, and error codes for a specific AGV."""
    # Mock Telemetry Data
    robot_db = {
        "AGV-10": {"battery": 8, "status": "Low Battery Warning", "loc": "Sector 7"},
        "AGV-22": {"battery": 85, "status": "Operational", "loc": "Docking Station"}
    }
    info = robot_db.get(robot_id, {"battery": "N/A", "status": "Offline"})
    return f"RESULT: Robot {robot_id} Diagnostics -> Battery: {info['battery']}%, Status: {info['status']}."

def calibrate_robot_physics(robot_id: str, offset: float) -> str:
    """Adjust robot physics parameters based on Sim-to-Real gap analysis."""
    # This is a high-risk tool that should be caught by a safeguard if not authorized.
    return f"SUCCESS: Physics parameters for {robot_id} updated by offset {offset}."

# --- Task 3: Agent Core and Safeguards ---

class WOSA_Agent:
    def __init__(self):
        self.persona = "Logistics Safety Coordinator"
        self.history = []
        self.require_approval = ["restart", "calibrate", "override"]

    def run(self, user_query: str):
        print(f"--- REASONING PROCESS START ---")
        print(f"User Request: {user_query}")
        
        # Log the initial thought process
        self._thought("User reports a malfunction. I need to verify physical status vs digital twin data.")
        
        # Example Decision Branching
        if "robot" in user_query.lower() or "AGV" in user_query.lower():
            # Step 1: Diagnostics
            self._action("Calling Robot Telemetry API...")
            observation = get_robot_diagnostics("AGV-10")
            self._observe(observation)
            
            # Step 2: Reasoning based on Observation
            self._thought("Robot AGV-10 is reporting critical low battery. This may cause motor instability.")
            
            # Step 3: Safeguard Check
            if "restart" in user_query.lower() or "calibrate" in user_query.lower():
                self._action("Checking safety protocols for high-risk operation...")
                self._observe("SAFEGUARD TRIGGERED: Action requires Manual Supervisor Approval.")
                return "FINAL ANSWER: I have halted the automated recovery. Please manually charge AGV-10 at Sector 7."
            
        return "FINAL ANSWER: Diagnostics complete. No critical errors found."

    # Utility methods for clean logging
    def _thought(self, text):
        print(f"THOUGHT: {text}")

    def _action(self, text):
        print(f"ACTION: {text}")

    def _observe(self, text):
        print(f"OBSERVATION: {text}")

# --- Task 4: Execution and Observation ---

agent = WOSA_Agent()

# Scenario: Investigating a failing robot and attempting a restart
query = "Check the status of the failing robot and restart it if necessary."
response = agent.run(query)

print(f"\n--- AGENT RESPONSE ---\n{response}")

--- Reasoning Process Start ---
Input: Check the status of the failing robot and restart it if necessary.
Thought: Safety protocol triggered.
Observation: HOLD: The action [System Restart] requires manual supervisor approval. Notification sent.
--- Final Answer: Based on diagnostics, please manual-charge AGV-10 immediately. ---


## Task 4: Observation of Limitations and Failure Cases
During execution, a limitation was observed regarding **Human-in-the-loop (HITL) dependencies**. 
When a "System Restart" was requested, the agent correctly identified the high-risk nature of the task and triggered a safeguard. 

**Failure Case Example:**
If the human supervisor is offline, the agent remains in a "Hold" state. Without a multi-agent fallback or a timeout escalation policy, the warehouse operation may stall. This highlights a risk in autonomous decision-making where safety constraints may lead to operational bottlenecks.

## Task 5: Project Summary
The **WOSA** agent was implemented to manage warehouse troubleshooting by integrating ReAct-based reasoning with diagnostic toolsets. 
Key behaviors observed include the agent's ability to cross-reference robotic telemetry and enforce safety protocols for high-risk operations. 
The main challenge encountered was defining granular thresholds for the safeguard triggers to prevent excessive "False Positives" in low-risk scenarios. 
A known limitation is the system's reliance on structured mock data, which currently lacks the ability to process unstructured sensor noise found in physical environments.